**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**05 | Classical Regression Baselines and Participant-Independent Validation**

![Verified Essex paired-data regression workflow, training-only GroupKFold and protected held-out assessment](../../images/05_consolidated_workflow.svg)

**Research question:** How accurately can transparent, pre-specified classical models predict the original eleven-position first and third principal femoral-neck FE strain curves from 42-channel wearable IMU trials, when the validation participants have not contributed to model fitting?

**Data foundation:** The original [Essex Figshare v13 dataset](https://doi.org/10.6084/m9.figshare.30225739.v13), with the verified participant/trial registry and source-exact time alignment recorded in [Notebook 04](04_governed_imu_fe_dataset_and_participant_splits.ipynb). Source inputs: 1,729 eligible trials, IMU (101×42), original FE targets (11×2); 168 original FE-empty trials are excluded.

**Study boundary:** Use the frozen Training (29 participants, 1,354 trials) and Validation (4 participants, 192 trials) split. The four reserved Test participants (183 trials) are never used for model fitting, CV, diagnostics, model choice or performance reporting in this notebook. Validation is diagnostic; **model selection is performed using Training-only grouped CV**.

**Target and statistical interpretation:** Published `e1` and `e3` are **unitless FE-model outputs**, not directly measured patient strain. Report original source-grid pointwise and peak errors, with each complete trial treated as one learning sample. Do not interpolate FE values, convert units, claim prospective clinical utility or compare these results with an independent external cohort.

**Execution:** The simplified cells have not been rerun on the full dataset. Numerical findings described here refer to the previously completed run. Each notebook reads the current upstream files; re-execution replaces its own output files. No historical reports, hash comparisons or PASSED gates are required.


**00. Colab Repository Setup**

Open an up-to-date, non-destructive checkout before importing research modules. The shared sync utility preserves local changes and ignored datasets.


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

Repository up to date.
/content/human-movement-intelligence


**01. Imports and Research Environment**

Use standard Colab paths, the pinned MATLAB reader, reusable Essex data recovery and regression utilities, and the shared scientific plotting style.


In [ ]:
# ==================== 01. Imports and Research Environment ===================

# ------------------------- 1. Study Environment Setup ------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5, label_weight="normal",
    tick_size=11, legend_size=10, figure_dpi=100, figure_size=(9, 4))

# ------------------- 5. Shared and Study Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from essex_timebase_audit import EXPECTED_CHANNELS
from essex_regression_baselines import (
    MODELS, TARGETS, trial_features, flatten_targets, make_regressor,
    training_group_folds, regression_metrics, participant_metrics,
    subject_balanced_mae, trial_predictions,
)
from essex_paired_dataset import read_trial_splits, load_paired_data

# ----------------------- 6. Source and Output Locations ----------------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
processed_dir = root / "data" / "processed" / "essex_femoral_neck_loading"
output_dir = study_dir / "output"
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Stage 05 evidence directory:", output_dir)


**02. Current Participant Splits and Model Inputs**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ======================= 02. Current Participant Splits ======================

# --------------------- 1. Read Current Trial Assignments ---------------------
assigned = read_trial_splits(output_dir)

# --------------------- 2. Prepare Participant Split Masks --------------------
train_mask = assigned["split"].eq("TRAIN").to_numpy()
valid_mask = assigned["split"].eq("VALIDATION").to_numpy()
test_mask = assigned["split"].eq("TEST").to_numpy()
display(assigned.groupby("split").agg(
    trials=("local_record_key", "size"),
    participants=("subject_id", "nunique"),
))


**03. Load Current Paired Arrays**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ======================= 03. Load Current Paired Arrays ======================

# --------------- 1. Install and Import the Pinned MATLAB Reader --------------
%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat

# ----------------------- 2. Load Current Paired Arrays -----------------------
paired = load_paired_data(output_dir, processed_dir, raw_dir, load_from_mat, assigned)

# ------------ 3. Select Arrays for the Required Participant Splits -----------
x_train_source = paired["X_full"][train_mask]
y_train_source = paired["y_fe"][train_mask]
x_valid_source = paired["X_full"][valid_mask]
y_valid_source = paired["y_fe"][valid_mask]
print("Loaded current arrays:", {name: value.shape for name, value in paired.items()})


**04. Trial-Level Features and Original FE Targets**

Represent each original 101-sample movement trial by five fixed, target-independent statistics for each of 42 IMU channels: mean, population standard deviation, minimum, maximum and RMS (210 predictors). Preserve 11 source FE positions × two targets as 22 outputs per trial.


In [ ]:
# ===================== 04. Trial Features and FE Targets =====================

# ----------------------- 1. Extract Fixed IMU Features -----------------------
x_train, feature_names = trial_features(
    x_train_source, EXPECTED_CHANNELS,
)
x_valid, confirmed_names = trial_features(
    x_valid_source, EXPECTED_CHANNELS,
)
y_train = y_train_source
y_valid = y_valid_source
z_train = flatten_targets(y_train)
z_valid = flatten_targets(y_valid)

# ----------------------- 2. Preserve Original Trial IDs ----------------------
train_info = assigned.loc[train_mask].reset_index(drop=True)
valid_info = assigned.loc[valid_mask].reset_index(drop=True)
feature_schema = pd.DataFrame({
    "feature_index": np.arange(210),
    "feature": feature_names,
    "statistic": np.repeat(
        ["mean", "std", "minimum", "maximum", "rms"], 42,
    ),
    "original_channel": list(EXPECTED_CHANNELS) * 5,
})
feature_schema.to_csv(
    output_dir / "05_feature_schema.csv", index=False,
)
print("Original trials as learning rows:", len(x_train))
print("Predictors:", len(feature_names), "| FE outputs per trial:", 22)
display(feature_schema.head(10))


**05. Pre-Specified Models and Training-Only Grouped Cross-Validation**

Use one fixed five-fold GroupKFold over the 29 Training participants. Each fold fits its own preprocessing, including a StandardScaler inside Ridge only. The three model configurations are fixed before any Validation scoring. Do not use Test to determine folds, parameters or predictions.


In [ ]:
# ================= 05. Training-Only Grouped Cross-Validation ================

# --------------------- 1. Fix Participant-Disjoint Folds ---------------------
CV_FOLDS = 5
SEED = 2026
train_groups = train_info["subject_id"].to_numpy()
folds, fold_ids = training_group_folds(
    train_groups, n_splits=CV_FOLDS,
)
fold_registry = train_info.loc[:, [
    "local_record_key", "subject_id", "task"
]].copy()
fold_registry["validation_fold"] = fold_ids
fold_registry.to_csv(
    output_dir / "05_training_fold_registry.csv", index=False,
)

# --------------- 2. Fit Fresh Estimators in Every Training Fold --------------
cv_rows = []
oof = {}
for name in MODELS:
    estimates = np.full(y_train.shape, np.nan, dtype=np.float64)
    for fold_number, (fit_rows, held_rows) in enumerate(folds, 1):
        learner = make_regressor(name, seed=SEED)
        learner.fit(x_train[fit_rows], z_train[fit_rows])
        pred = learner.predict(x_train[held_rows]).reshape(-1, 11, 2)
        estimates[held_rows] = pred
        fold_score = subject_balanced_mae(
            y_train[held_rows], pred, train_groups[held_rows]
        )
        for result in regression_metrics(y_train[held_rows], pred):
            cv_rows.append(
                {
                    "model": name,
                    "fold": fold_number,
                    "training_subjects": len(set(train_groups[fit_rows])),
                    "validation_subjects": len(set(train_groups[held_rows])),
                    "subject_balanced_mae": fold_score,
                    **result,
                }
            )
    oof[name] = estimates
cv_folds = pd.DataFrame(cv_rows)
cv_folds.to_csv(
    output_dir / "05_training_cv_fold_metrics.csv", index=False,
)
print("Completed Training-only grouped folds:", CV_FOLDS)
display(cv_folds.head(10))


**06. Cross-Validated Participant and Activity Errors**

Score every Training subject only from their out-of-fold predictions. Keep errors for e1 and e3 separate, report the subject-balanced score and choose the candidate model by Training CV alone.


In [ ]:
# ============== 06. Training Out-of-Fold Error and Model Choice ==============

# ------------------ 1. Source-Linked Participant Diagnostics -----------------
cv_person_tables = []
summary_rows = []
for name in MODELS:
    pred = oof[name]
    rows = participant_metrics(
        y_train, pred, train_info["subject_id"], train_info["task"],
        model=name, phase="TRAIN_OOF",
    )
    rows["fold"] = rows["subject_id"].map(
        fold_registry.drop_duplicates("subject_id").set_index(
            "subject_id"
        )["validation_fold"]
    ).astype(int)
    cv_person_tables.append(rows)
    score = subject_balanced_mae(y_train, pred, train_groups)
    summary_rows.append({
        "model": name,
        "subject_balanced_mae": score,
        "e1_oof_mae": regression_metrics(y_train, pred)[0]["mae"],
        "e3_oof_mae": regression_metrics(y_train, pred)[1]["mae"],
        "training_subjects": len(set(train_groups)),
        "training_trials": len(y_train),
        "cv_folds": CV_FOLDS,
    })
cv_person = pd.concat(cv_person_tables, ignore_index=True)
cv_summary = pd.DataFrame(summary_rows).sort_values(
    ["subject_balanced_mae", "model"], kind="stable",
).reset_index(drop=True)
selected_model = str(cv_summary.loc[0, "model"])
cv_person.to_csv(
    output_dir / "05_training_cv_subject_activity_metrics.csv",
    index=False,
)
cv_summary.to_csv(
    output_dir / "05_training_cv_model_summary.csv",
    index=False,
)
print("Selected on Training GroupKFold only:", selected_model)
display(cv_summary)


**07. Frozen Validation Evaluation**

After Training-only model selection, refit each of the three pre-specified models on the complete Training split. Evaluate all candidates on the same four Validation participants for transparent diagnostics; do not change the selected model based on these results.


In [ ]:
# ====================== 07. Frozen Validation Evaluation =====================

# ------------------- 1. Train on the Frozen Training Split -------------------
valid_predictions = {}
validation_metrics_rows = []
validation_long = []
for name in MODELS:
    learner = make_regressor(name, seed=SEED)
    learner.fit(x_train, z_train)
    prediction = learner.predict(x_valid).reshape(-1, 11, 2)
    valid_predictions[name] = prediction
    balanced = subject_balanced_mae(y_valid, prediction, valid_info["subject_id"])
    for item in regression_metrics(y_valid, prediction):
        validation_metrics_rows.append(
            {
                "model": name,
                "selected_from_training_cv": name == selected_model,
                "subject_balanced_mae": balanced,
                "validation_participants": 4,
                **item,
            }
        )
    validation_long.append(
        trial_predictions(
            y_valid,
            prediction,
            valid_info["local_record_key"],
            valid_info["subject_id"],
            valid_info["task"],
            name,
        )
    )
validation_metrics = pd.DataFrame(validation_metrics_rows)
validation_predictions = pd.concat(validation_long, ignore_index=True)
validation_metrics.to_csv(
    output_dir / "05_validation_model_metrics.csv", index=False,
)
validation_predictions.to_csv(
    output_dir / "05_validation_predictions.csv", index=False,
)
print("Training-CV-selected model:", selected_model)
print("Validation trials per candidate:", len(y_valid))
display(validation_metrics)


**08. Participant, Activity and Biomechanical Peak Analysis**

Report participant/activity heterogeneity without selecting models from Validation. MAE and RMSE aggregate error over the original FE coordinates; peak-value and peak-position errors use 11 source-grid positions, not interpolated time or external physiological calibration.


In [ ]:
# ================ 08. Participant, Activity and Peak Analysis ================

# ------------------- 1. Person- and Movement-Level Results -------------------
validation_details = pd.concat([
    participant_metrics(
        y_valid, valid_predictions[name],
        valid_info["subject_id"], valid_info["task"],
        model=name, phase="VALIDATION",
    ) for name in MODELS
], ignore_index=True)
validation_details.to_csv(
    output_dir / "05_validation_subject_activity_metrics.csv",
    index=False,
)
activity_report = validation_details.groupby(
    ["model", "task", "target"], as_index=False
).agg(
    mean_subject_activity_mae=("mae", "mean"),
    participants=("subject_id", "nunique"),
    trials=("trials", "sum"),
)
print("Validation error by movement and FE target:")
display(activity_report)
print("Peak index error uses native FE positions 0 to 10 only.")


**09. Scientific Comparison and Original-Grid Waveforms**

Create a grouped MAE comparison plot and two genuine FE waveform examples. Select the illustrated Validation trial by its source key rather than by prediction error; label it as an example, not a representative performance claim.


In [ ]:
# ======================= 09. Scientific Result Figures =======================

# ------------------ 1. Side-by-Side Validation MAE by Model ------------------
chart = validation_metrics.pivot(
    index="model", columns="target", values="mae",
).reindex(MODELS)
fig, ax = plt.subplots(figsize=(9, 4))
chart.plot.bar(ax=ax, width=0.75)
ax.set_title("Validation MAE by regression baseline and FE target")
ax.set_xlabel("Regression model")
ax.set_ylabel("MAE (published unitless FE scale)")
ax.tick_params(axis="x", rotation=15)
save_figure(fig, output_dir / "05_validation_model_comparison.png")
plt.show()

# ------------ 2. Fixed, Source-Identified Waveform Demonstrations ------------
example_index = int(np.argsort(
    valid_info["local_record_key"].to_numpy()
)[0])
example_key = str(valid_info.iloc[example_index]["local_record_key"])
print("Illustrative Validation trial:", example_key)
predicted = valid_predictions[selected_model][example_index]
observed = y_valid[example_index]
for j, target in enumerate(TARGETS):
    fig, ax = plt.subplots(figsize=(9, 4))
    position = np.arange(11)
    ax.plot(position, observed[:, j], marker="o", label="Source FE")
    ax.plot(position, predicted[:, j], marker="s",
            label=f"{selected_model} prediction")
    ax.set_title(f"Validation example: {target} | {example_key}")
    ax.set_xlabel("Original FE position (index 0–10)")
    ax.set_ylabel("Published unitless FE strain")
    ax.set_xticks(position)
    ax.legend()
    save_figure(
        fig, output_dir / f"05_validation_waveform_{target}.png",
    )
    plt.show()


**10. Results Summary and Output Download**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ================== 10. Results Summary and Output Download ==================

# ------------------------- 1. Display Current Results ------------------------
print("Current baseline results saved in output/ with prefix 05_.")
print("Training trials:", len(train_info), "| validation trials:", len(valid_info))

# -------------------- 2. Download Current Notebook Outputs -------------------
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="05",
)


**11. Scientific Interpretation and Next Research Boundary**

**Completed protocol:** Three pre-specified, trial-wise, multi-output regression baselines were trained with 210 target-independent IMU features and 22 source-aligned FE outputs (11 native time positions × `e1/e3`) per original trial. Five-fold participant-disjoint GroupKFold used only the 29 Training participants (1,354 trials). The candidate was chosen by equal-participant-weight, two-target out-of-fold MAE. **Random Forest** ranked first (**0.0007950**), followed by Ridge (**0.0012156**) and Training Mean (**0.0012562**). The four predefined Validation participants were not used for model selection.

**Recorded Validation results:** On four participants (192 trials), the Training-CV-selected Random Forest produced subject-balanced MAE **0.0006771** versus Ridge **0.0008758** and Training Mean **0.0012673**. Its `e1` waveform MAE/RMSE were **0.0003828 / 0.0005851**, while `e3` MAE/RMSE were **0.0009713 / 0.0015261**, respectively. These are errors on the original published **unitless FE-model** scale, not microstrain or clinical measurements. The lower aggregate Validation error than Training-CV error is descriptive and does not establish superior population generalisation.

**Biomechanical error trade-off:** Random Forest had the lowest average waveform error, but **Ridge had smaller Validation peak-value MAE for both targets**: `e1` **0.0006317** versus Random Forest **0.0006625**, and `e3` **0.0015181** versus **0.0016423**. Random Forest's mean peak-position absolute errors were **1.42 FE indices** for `e1` and **1.01 FE indices** for `e3`. Thus the model preferred for average waveform error is not automatically best for peak magnitude. Peak timing refers to the eleven **source FE coordinates**, not physical time measured in seconds. This comparison is an exploratory finding, **not** a reason to reselect the model using Validation.

**Next research boundary:** Investigate carefully budgeted temporal regression and robustness, with separate attention to pointwise waveform error and peak magnitude, using Training/Validation only. Preserve the predefined Test split for the final evaluation in Notebook 08.
